# 🛡️ Projeto CyberSentinel: Fine-Tuning do Analista SOC Virtual

**Objetivo da Atividade:** Realizar o ajuste fino (fine-tuning) de um LLM especialista em cibersegurança utilizando dados reais do HuggingFace. O modelo final será capaz de classificar ameaças, mapear logs para técnicas **MITRE ATT&CK**, avaliar impactos na tríade CIA (Confidencialidade, Integridade, Disponibilidade) e sugerir ações imediatas de resposta a incidentes.

**Abordagem Metodológica:**
- **Modelo Base:** `unsloth/Qwen3.5-2B-Instruct` (ou o modelo Qwen compatível de preferência).
- **Otimização:** QLoRA (Rank = 16, Alpha = 16) para redução drástica de uso de memória de GPU.
- **Dataset:** `sambanovasystems/attackqa` (dados de Q&A reais sobre MITRE ATT&CK).
- **Split de Dados:** 70% Treino e 30% Validação.
- **Portabilidade:** Fusão de adaptadores e exportação direta para **GGUF em 4 bits (q4_k_m)** para execução local em CPU no contêiner Docker.

## 1. Instalação de Dependências
Configuração do ambiente de GPU no Colab. O uso do instalador de pacotes rápido `uv` acelera muito a montagem das dependências.

In [5]:
# Instalação e atualização dos pacotes essenciais para processamento paralelo e fine-tuning com Unsloth
!pip install --upgrade -qqq uv
!uv pip install -qqq "torch==2.8.0" "triton>=3.3.0" torchvision bitsandbytes xformers==0.0.32.post2
!uv pip install -qqq "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" "unsloth[base] @ git+https://github.com/unslothai/unsloth"
!uv pip install -qqq --no-deps "torchcodec==0.7.0"
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2
!uv pip install transformers==5.2.0 flash-linear-attention causal_conv1d==1.6.0
!uv pip install --no-deps --upgrade "torchao>=0.16.0"

Using Python 3.12.13 environment at: /usr
Resolved 2 packages in 98ms
Checked 2 packages in 0.19ms
Using Python 3.12.13 environment at: /usr
Resolved 55 packages in 120ms
⠇ Preparing packages... (0/1)                                                   ^C
Using Python 3.12.13 environment at: /usr
Resolved 1 package in 124ms
Checked 1 package in 0.20ms


## 2. Inicialização do Modelo e Tokenizador
Carregamento do modelo base de texto `unsloth/Qwen3.5-2B-Instruct` habilitando quantização de 4 bits para economia de VRAM na GPU T4.

In [6]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 4096  # Suporte a janelas de contexto longas para logs e relatórios
dtype = None           # Detecção automática de tipo com base na GPU
load_in_4bit = True    # Ativa carregamento em 4 bits para viabilizar o treino em hardware modesto

model_id = "unsloth/Qwen3.5-2B"

print(f"Carregando o modelo base {model_id}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_id,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

Carregando o modelo base unsloth/Qwen3.5-2B...
==((====))==  Unsloth 2026.5.5: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.


Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

## 3. Parametrização PEFT (QLoRA)
Habilita o treinamento direcionado nas matrizes de atenção e projeção do modelo para reduzir os parâmetros treináveis e evitar o esquecimento catastrófico do modelo.

In [7]:
print("Configurando adaptadores LoRA (PEFT)...")
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,                # Rank do LoRA (ajuste fino de rank médio)
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha = 16,       # Fator de escala
    lora_dropout = 0,      # Desativado para fins de eficiência com Unsloth
    bias = "none",
    random_state = 42,
    use_rslora = False,
    loftq_config = None,
)

Configurando adaptadores LoRA (PEFT)...


## 4. Carregamento e Divisão do Dataset (70/30)
Aqui carregamos o dataset real `sambanovasystems/attackqa` e efetuamos a quebra científica em **70% para Treino** e **30% para Validação**.

In [9]:
from datasets import load_dataset

print("Carregando o dataset de segurança real do HuggingFace...")
dataset = load_dataset("sambanovasystems/attackqa", split="train")

# Definição do prompt sistemático que molda o comportamento do CyberSentinel
prompt_sistema = """Você é o CyberSentinel, um agente inteligente e analista SOC (Security Operations Center) sênior especialista em resposta a incidentes cibernéticos.
Você analisa logs, descrições de falhas, vulnerabilidades e alertas e fornece análises estruturadas detalhadas.
Ao receber um log ou incidente, estruture sua resposta RIGOROSAMENTE com as seguintes seções em Markdown:
1. ## 🔴 Classificação da Ameaça: Identifique o tipo de ataque e defina a severidade (CRITICAL, HIGH, MEDIUM, LOW ou INFO).
2. ## 🎯 Mapeamento MITRE ATT&CK: Indique a Tática e Técnica correspondente ao comportamento observado.
3. ## 🛡️ Análise de Impacto (CIA): Descreva brevemente o impacto à Confidencialidade, Integridade e Disponibilidade dos dados.
4. ## 📋 Plano de Resposta e Mitigação: Indique os passos práticos de contenção, erradicação e recuperação que o analista de infraestrutura deve executar."""

def preparar_dataset_conversacional(amostra):
    # Formata a entrada para o template de chat com ChatML/ShareGPT
    # Corrigido: o dataset usa 'answer' em vez de 'response'
    pergunta = amostra["question"]
    resposta = amostra["answer"]

    dialogo = [
        {"role": "system", "content": prompt_sistema},
        {"role": "user", "content": f"Analise a seguinte questão de segurança ou alerta:\n\n{pergunta}"},
        {"role": "assistant", "content": resposta}
    ]
    return {"messages": dialogo}

# Realizando a divisão 70/30 no dataset real
print("Dividindo o dataset em 70% Treino e 30% Validação...")
dataset_split = dataset.train_test_split(test_size=0.3, seed=42)

# Limitando o tamanho máximo para otimizar o tempo de treino no Colab gratuito (T4)
limite_treino = min(2100, len(dataset_split["train"]))
limite_val = min(900, len(dataset_split["test"]))

train_raw = dataset_split["train"].select(range(limite_treino))
val_raw = dataset_split["test"].select(range(limite_val))

print(f"Estruturando dados de Treinamento ({len(train_raw)} registros)...")
dataset_treino = [preparar_dataset_conversacional(a) for a in train_raw]

print(f"Estruturando dados de Validação ({len(val_raw)} registros)...")
dataset_validacao = [preparar_dataset_conversacional(a) for a in val_raw]

Carregando o dataset de segurança real do HuggingFace...
Dividindo o dataset em 70% Treino e 30% Validação...
Estruturando dados de Treinamento (2100 registros)...
Estruturando dados de Validação (900 registros)...


## 5. Loop de Treinamento e Otimização
Configuramos o `SFTTrainer` para rodar o treino com monitoramento de `eval_loss` a cada 20 passos.

In [10]:
from trl import SFTTrainer, SFTConfig

# Prepara modelo de linguagem para modo de treino
FastLanguageModel.for_training(model)

config_treino = SFTConfig(
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    warmup_steps = 10,
    max_steps = 120,           # Suficiente para alinhar no Colab em ~10 minutos
    learning_rate = 2e-4,
    logging_steps = 10,
    eval_strategy = "steps",   # Habilita avaliação contínua no processo
    eval_steps = 20,           # Calcula eval_loss a cada 20 passos
    optim = "adamw_8bit",      # Otimizador de 8 bits para VRAM reduzida
    weight_decay = 0.01,
    lr_scheduler_type = "linear",
    seed = 42,
    output_dir = "cybersentinel_checkpoints",
    report_to = "none",
    remove_unused_columns = False,
    dataset_text_field = "",
    dataset_kwargs = {"skip_prepare_dataset": True},
    max_seq_length = max_seq_length
)

In [23]:
from datasets import Dataset
from transformers import DataCollatorForLanguageModeling
from unsloth import get_chat_template
from trl import SFTTrainer

# 1. Transformamos em Dataset do Hugging Face
train_dataset = Dataset.from_list(dataset_treino)
eval_dataset = Dataset.from_list(dataset_validacao)

# 2. Configura o template de chat
# Pegamos o tokenizer base para garantir que não haja lógica visual interferindo
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "chatml",
)

# 3. Tokenização manual forçando apenas texto
def tokenize_function(examples):
    convos = examples["messages"]
    # Gera o texto final via template
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]

    # Tentamos acessar o tokenizer interno caso o objeto seja um Processor multimodal
    actual_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)

    # Tokeniza as strings usando o tokenizer de texto puro
    return actual_tokenizer(texts, truncation = True, padding = False)

print("Preparando datasets (tokenização de texto puro)...")
train_dataset = train_dataset.map(tokenize_function, batched = True, remove_columns = train_dataset.column_names)
eval_dataset = eval_dataset.map(tokenize_function, batched = True, remove_columns = eval_dataset.column_names)

# 4. Collator para preenchimento dinâmico
collator = DataCollatorForLanguageModeling(tokenizer = getattr(tokenizer, "tokenizer", tokenizer), mlm = False)

print("Iniciando o treinamento do CyberSentinel com QLoRA...")
trainer = SFTTrainer(
    model = model,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    data_collator = collator,
    max_seq_length = max_seq_length,
    tokenizer = getattr(tokenizer, "tokenizer", tokenizer),
    args = config_treino,
)

metricas_treino = trainer.train()
print("Treinamento finalizado com sucesso!")

Preparando datasets (tokenização de texto puro)...


Map:   0%|          | 0/2100 [00:00<?, ? examples/s]

Map:   0%|          | 0/900 [00:00<?, ? examples/s]

Iniciando o treinamento do CyberSentinel com QLoRA...
Unsloth: Switching to float32 training since model cannot work with float16


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,100 | Num Epochs = 1 | Total steps = 120
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 10,911,744 of 2,224,153,408 (0.49% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
20,0.571643,0.436102
40,0.371175,0.369570
60,0.323475,0.345258
80,0.326560,0.334654
100,0.315831,0.327627
120,0.374112,0.323334


Unsloth: Restored added_tokens_decoder metadata in cybersentinel_checkpoints/checkpoint-120/tokenizer_config.json.


Treinamento finalizado com sucesso!


## 6. Validação do Agente (Inferência)
Teste do modelo fine-tuned com um log real de tentativas de SSH Brute Force para validar a qualidade da formatação e precisão.

In [25]:
# Muda o modelo para modo de inferência
FastLanguageModel.for_inference(model)

teste_log = "Detectadas 450 tentativas falhas de login SSH para o usuário 'root' em menos de 3 minutos vindo do IP 192.168.1.150."

mensagens = [
    {"role": "system", "content": prompt_sistema},
    {"role": "user", "content": f"Analise a seguinte questão de segurança ou alerta:\n\n{teste_log}"}
]

actual_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)

# Tokenizamos garantindo o retorno da máscara de atenção
inputs = actual_tokenizer.apply_chat_template(
    mensagens,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict = True
).to("cuda")

print("Gerando resposta estruturada do CyberSentinel...")
saidas = model.generate(
    **inputs,
    max_new_tokens = 512,
    use_cache = True,
    temperature = 0.1, # Menor temperatura para maior rigor na estrutura
    top_p = 0.9,
    pad_token_id = actual_tokenizer.eos_token_id
)

# Decodifica removendo os tokens de prompt
resultado = actual_tokenizer.decode(saidas[0][len(inputs['input_ids'][0]):], skip_special_tokens=True)
print("\nResposta:\n")
print(resultado)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Gerando resposta do CyberSentinel...

Resposta:

The detection of 450 failed login attempts for the 'root' user from IP address 192.168.1.150 within a timeframe of less than 3 minutes indicates a potential brute force attack. An adversary may be attempting to gain unauthorized access to a system by guessing passwords or usernames. This could be a result of an adversary who has compromised a user account or has access to a password or username.


## 7. Exportação direta para GGUF (4 bits)
Unsloth suporta a fusão (merge) dos adaptadores LoRA aprendidos diretamente com o modelo base de 16 bits e a exportação direta em formato **GGUF quantizado de 4 bits (Q4_K_M)**.

O arquivo gerado é perfeito para colocarmos na nossa pasta do Docker local do backend (`/backend/models/model.gguf`) para execução em CPU.

In [26]:
print("Fundindo adaptadores e exportando modelo para GGUF...")
model.save_pretrained_gguf(
    "cybersentinel_gguf",
    tokenizer,
    quantization_method = "q4_k_m"
)
print("Exportação concluída! Baixe o arquivo '.gguf' gerado na pasta 'cybersentinel_gguf' e coloque-o na pasta 'backend/models' do Docker local renomando para 'model.gguf'.")

Fundindo adaptadores e exportando modelo para GGUF...
Unsloth: Merging model weights to 16-bit format...


Unsloth: Restored added_tokens_decoder metadata in cybersentinel_gguf/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...


Unsloth: Copying 1 files from cache to `cybersentinel_gguf`: 100%|██████████| 1/1 [03:38<00:00, 218.36s/it]


Successfully copied all 1 files from cache to `cybersentinel_gguf`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [04:24<00:00, 264.40s/it]


Unsloth: Merge process complete. Saved to `/content/cybersentinel_gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['cybersentinel_gguf_gguf/Qwen3.5-2B.F16.gguf', 'cybersentinel_gguf_gguf/Qwen3.5-2B.F16-mmproj.gguf']
Unsloth: [2] Converting GGUF f16 into q